## NLP & feature engineering

Unlike some of the other notebooks, the purpose of this one may not be immediately obvious from its name. In this notebook, I will:

- expand the technology dictionary to cover 100+ technologies
- perform TF-IDF analysis to identify the top 20 words that best distinguish high and low-salary vacancies
- classify job grades into **Junior**, **Middle**, **Senior**, and **Unknown**
- build a binary skill matrix and export it as `features_nlp.csv`

In [1]:
import pandas as pd
import numpy as np
import re
import ast
import plotly.graph_objects as go
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
from itertools import chain

In [2]:
DATE_STR = "2026-04-14"
FILEPATH = f"../data/processed/vacancies_clean_{DATE_STR}.csv"
OUTPUT_DIR = "../data/vizualizations/"

df_cleaned = pd.read_csv(FILEPATH)

# work_format and skills_extracted - strings, so we need to restore lists
df_cleaned["work_format"] = df_cleaned["work_format"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)
df_cleaned["skills_extracted"] = df_cleaned["skills_extracted"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else []
)

df_labeled = df_cleaned[df_cleaned["salary_mid"].notna()].copy()
df_unlabeled = df_cleaned[df_cleaned["salary_mid"].isna()].copy()

_dark = dict(template="plotly_dark", width=1000, height=500)

print(f"df_cleaned: {df_cleaned.shape}")
print(f"df_labeled: {df_labeled.shape}")
print(f"df_unlabeled: {df_unlabeled.shape}")

df_cleaned: (1176, 20)
df_labeled: (367, 20)
df_unlabeled: (809, 20)


In [3]:
SKILL_ALIAS = {
    "python": "Python", "python3": "Python", "python 3": "Python",
    "js": "JavaScript", "javascript": "JavaScript", "es6": "JavaScript", "es2015": "JavaScript",
    "ts": "TypeScript", "typescript": "TypeScript",
    "java": "Java",
    "go": "Go", "golang": "Go",
    "kotlin": "Kotlin",
    "swift": "Swift",
    "c++": "C++", "cpp": "C++",
    "c#": "C#", "csharp": "C#", "c sharp": "C#",
    "php": "PHP",
    "ruby": "Ruby",
    "scala": "Scala",
    "rust": "Rust",
    "r": "R", "r language": "R",
    "dart": "Dart",
    "bash": "Bash", "shell": "Bash", "shell script": "Bash", "bash script": "Bash",
    "sql": "SQL", "pl/sql": "SQL", "t-sql": "SQL", "tsql": "SQL",
    "groovy": "Groovy",
    "elixir": "Elixir",
    "lua": "Lua",
    "matlab": "MATLAB",
    "1c": "1C", "1с": "1C", "1с-разработчик": "1C",
    "django": "Django",
    "fastapi": "FastAPI", "fast api": "FastAPI",
    "flask": "Flask",
    "react": "React", "reactjs": "React", "react.js": "React",
    "vue": "Vue", "vuejs": "Vue", "vue.js": "Vue",
    "angular": "Angular", "angularjs": "Angular",
    "spring": "Spring", "spring boot": "Spring",
    ".net": ".NET", "dotnet": ".NET", "dot net": ".NET", "asp.net": ".NET",
    "node.js": "Node.js", "nodejs": "Node.js", "node js": "Node.js",
    "laravel": "Laravel",
    "nestjs": "NestJS", "nest.js": "NestJS",
    "express": "Express", "express.js": "Express", "expressjs": "Express",
    "next.js": "Next.js", "nextjs": "Next.js",
    "nuxt": "Nuxt.js", "nuxt.js": "Nuxt.js",
    "svelte": "Svelte",
    "flutter": "Flutter",
    "react native": "React Native",
    "gin": "Gin",
    "echo": "Echo",
    "grpc": "gRPC", "grpc framework": "gRPC",
    "graphql": "GraphQL",
    "celery": "Celery",
    "tensorflow": "TensorFlow", "tf": "TensorFlow",
    "pytorch": "PyTorch", "torch": "PyTorch",
    "keras": "Keras",
    "scikit-learn": "scikit-learn", "sklearn": "scikit-learn",
    "pandas": "Pandas",
    "numpy": "NumPy",
    "xgboost": "XGBoost",
    "lightgbm": "LightGBM", "lgbm": "LightGBM",
    "catboost": "CatBoost",
    "hugging face": "Hugging Face", "huggingface": "Hugging Face", "transformers": "Hugging Face",
    "aws": "AWS", "amazon web services": "AWS",
    "gcp": "GCP", "google cloud": "GCP", "google cloud platform": "GCP",
    "azure": "Azure", "microsoft azure": "Azure",
    "yandex cloud": "Yandex Cloud", "яндекс облако": "Yandex Cloud",
    "digitalocean": "DigitalOcean", "digital ocean": "DigitalOcean",
    "postgresql": "PostgreSQL", "postgres": "PostgreSQL",
    "mysql": "MySQL",
    "mongodb": "MongoDB", "mongo": "MongoDB",
    "redis": "Redis",
    "elasticsearch": "Elasticsearch", "elastic": "Elasticsearch",
    "cassandra": "Cassandra",
    "clickhouse": "ClickHouse",
    "ms sql": "MSSQL", "mssql": "MSSQL", "sql server": "MSSQL", "microsoft sql server": "MSSQL",
    "oracle": "Oracle", "oracle db": "Oracle",
    "sqlite": "SQLite",
    "dynamodb": "DynamoDB", "amazon dynamodb": "DynamoDB",
    "neo4j": "Neo4j",
    "influxdb": "InfluxDB",
    "rabbitmq": "RabbitMQ", "rabbit mq": "RabbitMQ",
    "docker": "Docker",
    "kubernetes": "Kubernetes", "k8s": "Kubernetes",
    "git": "Git",
    "ci/cd": "CI/CD", "cicd": "CI/CD", "gitlab ci": "CI/CD", "github actions": "CI/CD", "jenkins": "CI/CD",
    "kafka": "Kafka", "apache kafka": "Kafka",
    "airflow": "Airflow", "apache airflow": "Airflow",
    "spark": "Spark", "apache spark": "Spark",
    "terraform": "Terraform",
    "ansible": "Ansible",
    "grafana": "Grafana",
    "prometheus": "Prometheus",
    "nginx": "Nginx",
    "linux": "Linux", "ubuntu": "Linux", "centos": "Linux", "debian": "Linux",
    "helm": "Helm",
    "dbt": "dbt",
    "mlflow": "MLflow",
    "kubeflow": "Kubeflow",
    "rest": "REST", "rest api": "REST", "restful": "REST",
    "hadoop": "Hadoop", "apache hadoop": "Hadoop",
    "hive": "Hive", "apache hive": "Hive",
    "jira": "Jira", "atlassian jira": "Jira",
    "confluence": "Confluence",
    "elk": "ELK", "elk stack": "ELK", "logstash": "ELK", "kibana": "ELK",
    "vault": "Vault", "hashicorp vault": "Vault",
}

print(f"All keys in SKILL_ALIAS: {len(SKILL_ALIAS)}")
print(f"Unique canonic skills: {len(set(SKILL_ALIAS.values()))}")

All keys in SKILL_ALIAS: 175
Unique canonic skills: 97


In [4]:
# I used same logic as in my eda notebook
_sorted_aliases = sorted(SKILL_ALIAS.keys(), key=lambda x: len(x.split()), reverse=True)
_mw = [re.escape(k) for k in _sorted_aliases if " " in k or "/" in k]
_sw = [re.escape(k) for k in _sorted_aliases if " " not in k and "/" not in k]
_PATTERN = re.compile(
    r"(?i)(" + "|".join(_mw + [r"\b"+ k + r"\b" for k in _sw]) + r")"
)

def normalize_skill(skill):
    return SKILL_ALIAS.get(skill.lower().strip(), skill.strip())

def extract_skills(text):
    matches = _PATTERN.findall(text)
    seen = set()
    result = []
    for m in matches:
        normalized_skill = normalize_skill(m)
        if normalized_skill not in seen:
            seen.add(normalized_skill)
            result.append(normalized_skill)
    return result

In [5]:
# Skill_extracted but now with 100+ technologies
df_cleaned["skills_extracted"] = df_cleaned["description"].apply(extract_skills)

# to see the difference
all_skills = list(chain.from_iterable(df_cleaned["skills_extracted"]))
skill_counts = Counter(all_skills)

print(f"Unique skills found: {len(skill_counts)}")
print(f"Top 10 skills: {skill_counts.most_common(10)}")
print(f"\nRows with >= 1 skill: {(df_cleaned['skills_extracted'].str.len() > 0).sum()}")

Unique skills found: 91
Top 10 skills: [('SQL', 382), ('REST', 290), ('Python', 285), ('CI/CD', 274), ('Docker', 224), ('Git', 215), ('PostgreSQL', 212), ('1C', 175), ('Kubernetes', 164), ('Linux', 164)]

Rows with >= 1 skill: 985


In [6]:
# I will fit TF-IDF on all descriptions, because full corpus gives more stable IDF weights
tfidf = TfidfVectorizer(
    max_features=5_000,
    ngram_range=(1, 2),
    min_df=5,                # ignoring words from <5 vacancies this is a very rare, noise
    sublinear_tf=True,
    stop_words=None,         # we have english and russian descriptions, so here keep none stop words
)

X_tfidf = tfidf.fit_transform(df_cleaned["description"].fillna(""))
feature_names = np.array(tfidf.get_feature_names_out())

print(f"TF-IDF matrix: {X_tfidf.shape}")

TF-IDF matrix: (1176, 5000)


In [7]:
median_salary = df_labeled["salary_mid"].median()
print(f"Median of salary_mid: {median_salary:_.0f} KZT")

labeled_id = df_labeled.index
high_id = df_labeled[df_labeled["salary_mid"] >= median_salary].index
low_id = df_labeled[df_labeled["salary_mid"] < median_salary].index

# positions of that indexes in X_tfidf
cleaned_pos = {idx: pos for pos, idx in enumerate(df_cleaned.index)}

high_pos = [cleaned_pos[i] for i in high_id]
low_pos = [cleaned_pos[i] for i in low_id]

print(f"high (>= median): {len(high_pos)} vacancies")
print(f"low (< median): {len(low_pos)} vacancies")

Median of salary_mid: 520_443 KZT
high (>= median): 185 vacancies
low (< median): 182 vacancies


In [8]:
# difference here shows association with high salary
mean_high = np.asarray(X_tfidf[high_pos].mean(axis=0)).ravel()
mean_low = np.asarray(X_tfidf[low_pos].mean(axis=0)).ravel()
diff = mean_high - mean_low

top_n = 20
top_high_idx = np.argsort(diff)[-top_n:][::-1]
top_low_idx = np.argsort(diff)[:top_n]

topterms = pd.DataFrame({
    "term": np.concatenate([feature_names[top_high_idx], feature_names[top_low_idx]]),
    "diff": np.concatenate([diff[top_high_idx], diff[top_low_idx]]),
    "group": ["high salary"] * top_n + ["low salary"] * top_n,
})
topterms = topterms.sort_values("diff")

fig7 = go.Figure()

for group, color in [("high salary", "#02BE9E"), ("low salary", "#FD6363")]:
    subset = topterms[topterms["group"] == group]
    fig7.add_trace(go.Bar(
        x=subset["diff"],
        y=subset["term"],
        orientation="h",
        name=group,
        marker_color=color,
    ))

fig7.update_layout(
    title="Top 20 TF-IDF terms by salary category",
    xaxis_title="Mean TF-IDF difference (high - low)",
    yaxis_title="Term",
    barmode="overlay",
    **_dark,
)

fig7.add_annotation(
    xref="paper", yref="paper", x=0.98, y=0.98,
    text=f"Median split: {median_salary/1_000:.0f}K KZT",
    showarrow=False, font=dict(size=11), xanchor="right",
)

fig7.write_html(f"{OUTPUT_DIR}/viz_7.html")
fig7.write_image(f"{OUTPUT_DIR}/viz_7.png")
fig7.show()

In [9]:
GRADE_KEYWORDS = {
    "Junior": [
        "junior", "jun", "jr", "джуниор", "джун",
        "стажер", "стажёр", "trainee", "intern", "internship", "начинающий", 
        "entry level", "entry-level", "strong junior"
    ],
    "Middle": [
        "middle", "mid level", "mid-level", "мидл", "middle+", "mid+", "strong middle", "опытный разработчик"
    ],
    "Senior": [
        "senior", "sen", "sr", "сеньор", "сениор", "старший", "ведущий", "principal", "architect", "архитектор",
        "staff engineer", "tech lead", "tech-lead", "team lead", "team-lead", "lead developer"
    ],
}

def normalize_text(text):
    text = str(text or "").lower()
    text = text.replace("+", " plus ")
    text = text.replace("/", " ")
    text = text.replace("-", " ")
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def build_pattern(words):
    escaped = [re.escape(w) for w in words]
    return re.compile(r"\b(" + "|".join(escaped) + r")\b", re.IGNORECASE)

GRADE_PATTERNS = {
    grade: build_pattern(words)
    for grade, words in GRADE_KEYWORDS.items()
}

GRADE_PRIORITY = {"Senior": 3, "Middle": 2, "Junior": 1}    #tiebreaker

def classify_grade(row):
    name = normalize_text(row.get("name", ""))
    desc = normalize_text(row.get("description", ""))
    exp = row.get("experience", "")
    score = {"Junior": 0, "Middle": 0, "Senior": 0}

    for grade in score:
        # name>desc
        if GRADE_PATTERNS[grade].search(name):
            score[grade] += 3
        if GRADE_PATTERNS[grade].search(desc):
            score[grade] += 1

    if exp == "noExperience":
        score["Junior"] += 2
    elif exp == "between1And3":
        score["Junior"] += 1
        score["Middle"] += 1
    elif exp == "between3And6":
        score["Middle"] += 2
        score["Senior"] += 1
    elif exp == "moreThan6":
        score["Senior"] += 2

    if not any(score.values()):
        return "Unknown"
    top_score = max(score.values())
    tied = [g for g, s in score.items() if s == top_score]

    return max(tied, key=lambda x: GRADE_PRIORITY[x])

df_cleaned["grade"] = df_cleaned.apply(classify_grade, axis=1)
print(df_cleaned["grade"].value_counts())

grade
Middle    833
Senior    249
Junior     94
Name: count, dtype: int64


In [10]:
ALL_SKILLS = sorted(set(SKILL_ALIAS.values()))

# binary matrix - 1 if skill in desc, 0 otherwise
skill_matrix = pd.DataFrame(
    [{skill: int(skill in skills) for skill in ALL_SKILLS} for skills in df_cleaned["skills_extracted"]],
    index=df_cleaned.index,
)

features = df_cleaned[[
    "id", "area", "employment", "experience", "has_usd_in_description",
    "salary_mid",          # nan for unlabeled, model will try to predict it
    "grade"]].copy()

WORK_FORMATS = ["ON_SITE", "REMOTE", "HYBRID", "FIELD_WORK"]
for format in WORK_FORMATS:
    features[f"wf_{format}"] = df_cleaned["work_format"].apply(lambda x: int(format in x))

feat_nlp = pd.concat([features.reset_index(drop=True), skill_matrix.reset_index(drop=True)], axis=1)

In [11]:
top3sk = skill_matrix.sum().sort_values(ascending=False).head(3).index.tolist()
grade_dist = feat_nlp["grade"].value_counts().to_dict()

print(
    f"Number of rows: {len(feat_nlp)}\n"
    f"Number of features: {feat_nlp.shape[1]}\n"
    f"Skills in matrix: {len(ALL_SKILLS)}\n"
    f"Top 3 skills: {', '.join(top3sk)}\n"
    f"Grades: {grade_dist}"
)

Number of rows: 1176
Number of features: 108
Skills in matrix: 97
Top 3 skills: SQL, REST, Python
Grades: {'Middle': 833, 'Senior': 249, 'Junior': 94}


In [12]:
out = f"../data/processed/features_nlp_{DATE_STR}.csv"
feat_nlp.to_csv(out, index=False)